# ⚡ Databricks
> Everything you need to know, explained short + sharp with examples

---

## 1. 🏠 Data Lakehouse

**What it is:** Combines cheap storage of a Data Lake + reliability/performance of a Data Warehouse.

```
Data Lake        → cheap storage, all formats, no ACID, no reliability
Data Warehouse   → fast queries, structured only, expensive
Data Lakehouse   → best of both → Delta Lake makes this possible
```

**One line answer:** *"A Lakehouse stores raw to refined data in one platform with ACID transactions, schema enforcement and BI + ML support — powered by Delta Lake on Databricks."*

---

## 2. 🏅 Medallion Architecture

**What it is:** 3-layer data organization pattern inside a Lakehouse.

```
Raw Files → Bronze (raw) → Silver (clean) → Gold (business-ready) → BI / ML
```

| Layer | What it holds | Key rule |
|---|---|---|
| **Bronze** | Raw data as-is, no changes | Never transform here |
| **Silver** | Cleaned, validated, joined | Trusted, deduplicated |
| **Gold** | Aggregated, business metrics | Optimized for dashboards/ML |

**If asked "how would you design this pipeline?"** → Always answer with Medallion.

---

## 3. 🔷 Delta Lake

**What it is:** Storage layer that adds database-like features on top of Parquet files.

### Why Delta over plain Parquet?

| Feature | Plain Parquet | Delta Lake |
|---|---|---|
| ACID Transactions | ❌ | ✅ |
| Updates / Deletes | ❌ | ✅ |
| Time Travel | ❌ | ✅ |
| Schema Enforcement | ❌ | ✅ |
| Streaming + Batch | ❌ | ✅ |

### ACID Transactions:
```
A → Atomicity   : all or nothing (no partial writes)
C → Consistency : data always valid
I → Isolation   : concurrent writes don't conflict
D → Durability  : committed data is never lost
```

### Time Travel:
```sql
-- Query older version by version number
SELECT * FROM gizmobox.silver.customers VERSION AS OF 3;

-- Query by timestamp
SELECT * FROM gizmobox.silver.customers TIMESTAMP AS OF '2024-01-01';

-- See all versions
DESCRIBE HISTORY gizmobox.silver.customers;
```

### OPTIMIZE & VACUUM:
```sql
-- Compact small files into larger ones → faster reads
OPTIMIZE gizmobox.silver.customers;

-- Add Z-ordering for even faster filtering on specific columns
OPTIMIZE gizmobox.silver.customers ZORDER BY (customer_id);

-- Delete old data files no longer needed (default 7 days retention)
VACUUM gizmobox.silver.customers;

-- Force delete files older than 0 hours (careful in production!)
VACUUM gizmobox.silver.customers RETAIN 0 HOURS;
```

### Managed vs External Tables:

| | Managed | External |
|---|---|---|
| **Storage** | Databricks controls location | You specify the path |
| **Delete table** | Data + metadata deleted | Only metadata deleted, data stays |
| **Format** | Always Delta | Any format |
| **Use when** | Standard pipelines | Data shared with other systems |

```sql
-- Managed table (Databricks manages everything)
CREATE TABLE gizmobox.bronze.customers
USING DELTA
AS SELECT * FROM json.`/path/to/files/`;

-- External table (you control the path)
CREATE TABLE gizmobox.bronze.customers
USING DELTA
LOCATION 'abfss://container@storage.dfs.core.windows.net/bronze/customers/';
```

---

## 4. 🔄 MERGE INTO (Upsert) — Most Important SQL

**What it is:** Insert new records + update existing ones in a single operation. Used everywhere in Silver layer.

```sql
MERGE INTO gizmobox.silver.customers AS target
USING gizmobox.bronze.customers AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
  UPDATE SET
    target.name    = source.name,
    target.email   = source.email,
    target.updated = source.updated

WHEN NOT MATCHED THEN
  INSERT (customer_id, name, email, updated)
  VALUES (source.customer_id, source.name, source.email, source.updated);
```

**Real use case:** New customer data arrives daily. Some are new customers (INSERT), some updated their email (UPDATE) → MERGE handles both in one go.

---

## 5. 🪟 Window Functions

**What it is:** Perform calculations across a group of rows without collapsing them (unlike GROUP BY).

```sql
-- ROW_NUMBER — rank rows within each group
SELECT
  customer_id,
  order_id,
  order_date,
  ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS row_num
FROM gizmobox.silver.orders;

-- Get latest order per customer
SELECT * FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS row_num
  FROM gizmobox.silver.orders
) WHERE row_num = 1;

-- Other useful window functions
RANK()        OVER (PARTITION BY ... ORDER BY ...)  -- rank with gaps
DENSE_RANK()  OVER (PARTITION BY ... ORDER BY ...)  -- rank without gaps
SUM(amount)   OVER (PARTITION BY customer_id)       -- running total per customer
LAG(column)   OVER (PARTITION BY ... ORDER BY ...)  -- previous row value
LEAD(column)  OVER (PARTITION BY ... ORDER BY ...)  -- next row value
```

---

## 6. 📝 CTEs (Common Table Expressions)

**What it is:** Named temporary result set inside a query — makes complex SQL readable.

```sql
-- Without CTE (messy)
SELECT * FROM (SELECT customer_id, COUNT(*) as order_count FROM orders GROUP BY customer_id) t
WHERE t.order_count > 5;

-- With CTE (clean)
WITH order_counts AS (
  SELECT customer_id, COUNT(*) AS order_count
  FROM gizmobox.silver.orders
  GROUP BY customer_id
),
high_value_customers AS (
  SELECT customer_id
  FROM order_counts
  WHERE order_count > 5
)
SELECT c.name, c.email, o.order_count
FROM gizmobox.silver.customers c
JOIN order_counts o ON c.customer_id = o.customer_id
WHERE c.customer_id IN (SELECT customer_id FROM high_value_customers);
```

---

## 7. 🐍 PySpark Transformations — Know These Cold

```python
# ── Read ──────────────────────────────────────────────────────
df = spark.read.format("delta").table("gizmobox.silver.customers")
df = spark.read.json("/path/")
df = spark.read.option("header", True).option("sep", "\t").csv("/path/")

# ── Select & Filter ───────────────────────────────────────────
df.select("customer_id", "name", "email")
df.filter(df.country == "IN")
df.where("order_amount > 100")

# ── Add / Rename / Drop Columns ───────────────────────────────
from pyspark.sql.functions import col, upper, lit, current_timestamp

df.withColumn("name_upper", upper(col("name")))
df.withColumn("ingested_at", current_timestamp())
df.withColumnRenamed("old_name", "new_name")
df.drop("unwanted_column")

# ── Aggregations ──────────────────────────────────────────────
df.groupBy("country").agg(
    count("customer_id").alias("total_customers"),
    sum("order_amount").alias("total_revenue"),
    avg("order_amount").alias("avg_order")
)

# ── Joins ─────────────────────────────────────────────────────
customers_df.join(orders_df, on="customer_id", how="left")
customers_df.join(orders_df, on="customer_id", how="inner")
customers_df.join(orders_df, on="customer_id", how="full")

# ── Clean Data ────────────────────────────────────────────────
df.dropDuplicates(["customer_id"])
df.dropDuplicates()
df.fillna({"email": "unknown", "age": 0})
df.drop("unwanted_col")
df.filter(col("customer_id").isNotNull())

# ── Write ─────────────────────────────────────────────────────
df.write.format("delta").mode("overwrite").saveAsTable("gizmobox.bronze.customers")
df.write.format("delta").mode("append").saveAsTable("gizmobox.bronze.customers")
df.write.format("delta").mode("overwrite").partitionBy("year","month").save("/path/")
```

---

## 8. 🔐 Unity Catalog

**What it is:** Centralized governance layer for all data assets across all workspaces.

```
3-level namespace:  catalog . schema . table
Example:            gizmobox . silver . customers
```

```sql
SHOW CATALOGS;
CREATE CATALOG IF NOT EXISTS gizmobox;
USE CATALOG gizmobox;

SHOW SCHEMAS IN gizmobox;
CREATE SCHEMA IF NOT EXISTS gizmobox.silver;

-- Grant access
GRANT SELECT ON TABLE gizmobox.gold.sales TO `analyst-group`;
GRANT MODIFY ON SCHEMA gizmobox.silver TO `engineer-group`;
REVOKE SELECT ON TABLE gizmobox.gold.sales FROM `analyst-group`;
```

**Key concepts:**
- Managed tables → Databricks manages storage, Delta only
- External tables → you manage path, any format
- Volumes → for storing files (not tables)
- Unity Catalog replaces the old per-workspace Hive Metastore

---

## 9. 📥 Auto Loader

**What it is:** Incrementally ingests new files as they arrive in cloud storage — no duplicates, no reprocessing.

```python
# Auto Loader — standard Bronze ingestion pattern
df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/raw/_schema/customers") \
    .load("/Volumes/gizmobox/landing/operational/customers/")

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/gizmobox/landing/raw/_checkpoint/customers") \
    .trigger(availableNow=True) \
    .toTable("gizmobox.bronze.customers")
```

**Key points:**
- Uses `cloudFiles` format — tracks which files already processed
- `schemaLocation` — saves inferred schema so it doesn't re-infer every run
- `checkpointLocation` — mandatory, tracks streaming progress
- Best practice for Bronze layer file ingestion in production

---

## 10. 🌊 Structured Streaming

**What it is:** Process continuous/real-time data as micro-batches.

```python
# Read stream
stream_df = spark.readStream \
    .format("delta") \
    .table("gizmobox.bronze.customers")

# Transform (same as batch!)
from pyspark.sql.functions import upper
transformed = stream_df.withColumn("name", upper(col("name")))

# Write stream
transformed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/checkpoint/silver_customers") \
    .trigger(processingTime="1 minute") \
    .toTable("gizmobox.silver.customers")
```

**Output modes:**
- `append` — only new rows written (most common)
- `complete` — entire result rewritten every batch
- `update` — only changed rows written

**⚠️ Always set `checkpointLocation`** — without it, stream loses state on restart and reprocesses everything from start.

---

## 11. 🔁 DLT (Delta Live Tables / Lakeflow) — Declarative Pipelines

**What it is:** You declare WHAT you want, Databricks figures out HOW to run it — handles dependencies, retries, data quality automatically.

```python
import dlt
from pyspark.sql.functions import *

# Bronze — raw ingestion
@dlt.table(
  name="bronze_customers",
  comment="Raw customers from landing zone"
)
def bronze_customers():
    return spark.readStream \
        .format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .load("/Volumes/gizmobox/landing/operational/customers/")

# Silver — cleaned with data quality checks
@dlt.table(name="silver_customers")
@dlt.expect("valid_customer_id", "customer_id IS NOT NULL")        # log bad rows
@dlt.expect_or_drop("valid_email", "email IS NOT NULL")            # drop bad rows
@dlt.expect_or_fail("no_duplicates", "COUNT(*) = COUNT(DISTINCT customer_id)")  # stop pipeline
def silver_customers():
    return dlt.read_stream("bronze_customers") \
        .withColumn("name", initcap(col("name"))) \
        .dropDuplicates(["customer_id"])
```

**3 data quality expectations:**

| Expectation | What happens to bad rows |
|---|---|
| `@dlt.expect` | Logged as warning, rows kept |
| `@dlt.expect_or_drop` | Bad rows dropped silently |
| `@dlt.expect_or_fail` | Pipeline stops completely |

---

## 12. 📊 SCD — Slowly Changing Dimensions

**What it is:** Strategy for handling historical changes in dimension data (customers, products etc.)

| Type | What it does | Example |
|---|---|---|
| **SCD Type 1** | Overwrite old value | Customer changes email → just update it |
| **SCD Type 2** | Keep history with start/end dates | Customer moves city → keep old row + add new row |

```sql
-- SCD Type 2 pattern using MERGE
MERGE INTO gizmobox.silver.customers AS target
USING new_data AS source
ON target.customer_id = source.customer_id
  AND target.is_current = true

WHEN MATCHED AND target.city != source.city THEN
  UPDATE SET target.is_current = false, target.end_date = current_date()

WHEN NOT MATCHED THEN
  INSERT (customer_id, city, is_current, start_date, end_date)
  VALUES (source.customer_id, source.city, true, current_date(), null);
```

---

## 13. 🔄 APPLY CHANGES API (DLT) — SCD in Databricks

**What it is:** Databricks-native way to implement SCD using DLT — much simpler than writing MERGE manually.

```python
import dlt

# Step 1 — create target table
dlt.create_streaming_table("silver_customers")

# Step 2 — apply changes (handles SCD automatically)
dlt.apply_changes(
    target = "silver_customers",          # target table
    source = "bronze_customers",          # source stream
    keys   = ["customer_id"],             # unique key
    sequence_by = col("updated_at"),      # which record is latest

    # SCD Type 1 (default) — just overwrite
    # stored_as_scd_type = 1

    # SCD Type 2 — keep full history
    stored_as_scd_type = 2
)
```

**Why use APPLY CHANGES over manual MERGE?**
- No boilerplate MERGE code to write
- Handles out-of-order records automatically (uses `sequence_by`)
- SCD Type 1 and Type 2 with one parameter change
- Built-in with DLT — no extra logic needed

---

## 14. 👁️ Views — 3 Types (Quick Reference)

```sql
-- Permanent View (always available, stored in catalog)
CREATE OR REPLACE VIEW gizmobox.bronze.v_customers AS
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`;

-- Temporary View (current session only)
CREATE OR REPLACE TEMPORARY VIEW tv_customers AS
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`;

-- Global Temp View (all sessions, until Spark app ends)
CREATE OR REPLACE GLOBAL TEMP VIEW gtv_customers AS
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`;

-- Query global temp view
SELECT * FROM global_temp.gtv_customers;

-- Streaming view (live, always up to date)
CREATE OR REPLACE STREAMING VIEW sv_customers AS
SELECT * FROM STREAM(gizmobox.bronze.customers);
```

---

## 15. 🖥️ Cluster Types (Quick Recap)

| Cluster | Use for | Key point |
|---|---|---|
| **All-purpose** | Development, notebooks, exploration | Expensive if left running |
| **Job Cluster** | Production pipelines, scheduled jobs | Starts for job, terminates after → cost efficient |
| **SQL Warehouse** | Analysts running SQL, BI tools | Serverless, scales to zero |

**Access modes for production:** Always **Shared** mode → multiple users, isolation, Unity Catalog support.

---

## 16. ⏰ Workflows / Jobs (Quick Recap)

**What it is:** Schedule and orchestrate notebooks, DLT pipelines, Python scripts.

- **Job Cluster** always for production — not all-purpose
- Set **retry policies** for failure handling
- Use **task dependencies** for multi-step pipelines
- Trigger types: Scheduled (cron), File arrival, Manual

---

## 🎯 If Asked in Assessment — Cheat Sheet Answers

| Question | Answer in 1-2 lines |
|---|---|
| "What is a Lakehouse?" | Data Lake + Data Warehouse — cheap storage + ACID reliability via Delta Lake |
| "What is Medallion Architecture?" | Bronze (raw) → Silver (clean) → Gold (business-ready) — layered pipeline pattern |
| "Why Delta over Parquet?" | ACID transactions, time travel, updates/deletes, schema enforcement |
| "What is MERGE INTO?" | Upsert — insert new rows + update existing ones in one operation |
| "What is Auto Loader?" | Incremental file ingestion using cloudFiles format — no duplicates, schema inference |
| "What is Unity Catalog?" | Centralized governance — catalog.schema.table namespace, permissions, lineage, audit |
| "What is DLT?" | Declarative pipeline framework — declare tables with @dlt.table, UC handles orchestration |
| "What is APPLY CHANGES?" | DLT's built-in SCD handler — SCD Type 1 or 2 with one parameter, no manual MERGE |
| "Job cluster vs All-purpose?" | Job cluster for production (cost-efficient, auto-terminates), all-purpose for dev |
| "What is checkpointLocation?" | Tracks streaming progress — mandatory, without it stream reprocesses everything on restart |

---

*Use this as your quick-scan reference before the assessment and during onboarding 🚀*

# ⚡ Databricks Quick Cheat Sheet

---

## 🏠 Lakehouse & Medallion
```
Lakehouse = Data Lake (cheap storage) + Data Warehouse (reliability) → powered by Delta Lake

Bronze  → Raw data as-is, no changes
Silver  → Cleaned, validated, deduplicated
Gold    → Business aggregates, ready for BI/ML
```

---

## 🔷 Delta Lake
```sql
-- Time Travel
SELECT * FROM table VERSION AS OF 3;
SELECT * FROM table TIMESTAMP AS OF '2024-01-01';
DESCRIBE HISTORY table;

-- Optimize & Clean
OPTIMIZE table ZORDER BY (customer_id);
VACUUM table RETAIN 168 HOURS;  -- 7 days default
```

---

## 🔄 MERGE (Upsert)
```sql
MERGE INTO silver.customers AS target
USING bronze.customers AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
  UPDATE SET target.name = source.name
WHEN NOT MATCHED THEN
  INSERT *;
```

---

## 🪟 Window Functions
```sql
ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC)
RANK()        OVER (PARTITION BY country    ORDER BY revenue DESC)
SUM(amount)   OVER (PARTITION BY customer_id)
LAG(amount)   OVER (PARTITION BY customer_id ORDER BY order_date)
```

---

## 📝 CTE
```sql
WITH cleaned AS (
  SELECT * FROM bronze.customers WHERE customer_id IS NOT NULL
),
ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated DESC) AS rn
  FROM cleaned
)
SELECT * FROM ranked WHERE rn = 1;
```

---

## 🐍 PySpark — Must Know
```python
# Read
df = spark.read.format("delta").table("gizmobox.silver.customers")

# Transform
df.select("id", "name")
df.filter(col("country") == "IN")
df.withColumn("name_upper", upper(col("name")))
df.withColumnRenamed("old", "new")
df.groupBy("country").agg(count("id").alias("total"))
df.join(orders_df, on="customer_id", how="left")
df.dropDuplicates(["customer_id"])
df.fillna({"email": "unknown"})

# Write
df.write.format("delta").mode("overwrite").saveAsTable("gizmobox.bronze.customers")
df.write.format("delta").mode("append").saveAsTable("gizmobox.bronze.customers")
```

---

## 🔍 Direct File Query
```sql
-- Single folder (always use folder not single file)
SELECT * FROM json.`/Volumes/gizmobox/landing/operational/customers/`
SELECT * FROM csv.`/path/`
SELECT * FROM parquet.`/path/`
-- ↑ backticks not quotes!
```

---

## 👁️ Views
```sql
CREATE OR REPLACE VIEW gizmobox.bronze.v_customers AS SELECT * FROM json.`/path/`;
CREATE OR REPLACE TEMPORARY VIEW tv_customers AS SELECT ...;        -- session only
CREATE OR REPLACE GLOBAL TEMP VIEW gtv_customers AS SELECT ...;    -- app only
SELECT * FROM global_temp.gtv_customers;                           -- query global
```

---

## 🔐 Unity Catalog
```sql
-- Namespace:  catalog.schema.table
SHOW CATALOGS;
CREATE CATALOG IF NOT EXISTS gizmobox;
USE CATALOG gizmobox;
CREATE SCHEMA IF NOT EXISTS gizmobox.bronze;
GRANT SELECT ON TABLE gizmobox.gold.sales TO `analysts`;
REVOKE SELECT ON TABLE gizmobox.gold.sales FROM `analysts`;
```

---

## 📥 Auto Loader
```python
df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", "/path/_schema") \
    .load("/Volumes/gizmobox/landing/operational/customers/")

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/path/_checkpoint") \
    .trigger(availableNow=True) \
    .toTable("gizmobox.bronze.customers")
```

---

## 🌊 Structured Streaming
```python
stream_df = spark.readStream.format("delta").table("bronze.customers")

stream_df.writeStream \
    .format("delta") \
    .outputMode("append")        # append / complete / update
    .option("checkpointLocation", "/checkpoint/path") \
    .trigger(processingTime="1 minute") \
    .toTable("silver.customers")
```

---

## 🔁 DLT (Delta Live Tables)
```python
import dlt

@dlt.table(name="bronze_customers", comment="Raw ingestion")
def bronze_customers():
    return spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .load("/path/customers/")

@dlt.table(name="silver_customers")
@dlt.expect("valid_id",    "customer_id IS NOT NULL")   # keep + warn
@dlt.expect_or_drop("valid_email", "email IS NOT NULL") # drop bad rows
def silver_customers():
    return dlt.read_stream("bronze_customers") \
        .dropDuplicates(["customer_id"])
```

---

## 🔄 APPLY CHANGES (SCD in DLT)
```python
dlt.create_streaming_table("silver_customers")

dlt.apply_changes(
    target       = "silver_customers",
    source       = "bronze_customers",
    keys         = ["customer_id"],
    sequence_by  = col("updated_at"),
    stored_as_scd_type = 2   # 1 = overwrite, 2 = keep history
)
```

---

## 🖥️ Clusters
```
All-purpose cluster  → dev/notebooks (expensive if left on)
Job cluster          → production pipelines (auto-terminates after job ✅)
SQL Warehouse        → analysts/BI tools (serverless, scales to zero ✅)
Shared access mode   → multiple users, Unity Catalog support, production ✅
```

---

## 🎯 One-Line Answers
| Topic | Say this |
|---|---|
| Lakehouse | Lake storage + warehouse reliability via Delta Lake |
| Medallion | Bronze (raw) → Silver (clean) → Gold (business-ready) |
| Delta vs Parquet | ACID, time travel, updates/deletes, schema enforcement |
| MERGE | Insert new + update existing in one operation |
| Auto Loader | Incremental file ingestion, no duplicates, cloudFiles format |
| Unity Catalog | catalog.schema.table — centralized governance + permissions |
| DLT | Declare tables with @dlt.table, Databricks handles orchestration |
| APPLY CHANGES | DLT's built-in SCD — Type 1 overwrite or Type 2 full history |
| checkpointLocation | Tracks streaming state — mandatory, always set it |
| Job vs All-purpose | Job cluster for prod (cheap), all-purpose for dev |